In [1]:
# input
pdb_fasta = "./tmp/pdb_metal_anno.fasta" # from https://github.com/wangchulab/MetalNet2/blob/main/dataset/collect/cmd.sh step 1, mbps.fasta
new_pdb_fasta = "../../../pdb/collect_mbp/tmp/mbp.fasta"
uniprot_fasta = "./tmp/uniprot_metal_anno.fasta"
anno_site_file = "./tmp/pdb_metal_anno.tsv"
# output
uni_anno_file = "./data/entryId-seqNum-resi-metalResi-pdbId.tsv"

In [2]:
pdb_id_to_seq = dict()
uni_id_to_seq = dict()
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

for r in SeqIO.parse(pdb_fasta, "fasta"):
    r: SeqRecord
    pdb_id_to_seq[str.upper(r.id)] = str(r.seq)
for r in SeqIO.parse(new_pdb_fasta, "fasta"):
    r: SeqRecord
    pdb_id_to_seq[str.upper(r.id)] = str(r.seq)
for r in SeqIO.parse(uniprot_fasta, "fasta"):
    r: SeqRecord
    uni_id_to_seq[r.id.split("-")[1]] = str(r.seq)

In [3]:
from Bio import Align

def align_two_seq(seq1, seq2) -> dict: # from https://github.com/wangchulab/MetalNet2/blob/main/dataset/collect/scripts/utils/mmcif_utils.py
    if seq1 == seq2:
        return dict(zip(range(len(seq1)), range(len(seq2))))

    # align two peptide sequence
    aligner = Align.PairwiseAligner(scoring="blastp")
    aligns = aligner.align(seq1, seq2)
    align = aligns[0].format().strip().split(
        "\n")  # take the first align result
    seq1, seq2 = '', ''
    for i in align:
        if i.startswith("target"):
            try:
                seq1 += i.split()[2]
            except:
                continue  # when the new line has no sequence, e.g., 'target 56'
        elif i.startswith("query"):
            try:
                seq2 += i.split()[2]
            except:
                continue

    # generate path dict: index from seq1 to index from seq2 (matched or replaced)
    seq1_to_seq2 = dict()
    seq1_index, seq2_index = 0, 0
    for i in range(len(seq1)):
        if seq1[i] != "-":
            if seq2[i] != "-":
                seq1_to_seq2[seq1_index] = seq2_index
            seq1_index += 1
        if seq2[i] != "-":
            seq2_index += 1
    return seq1_to_seq2

In [4]:
import pandas as pd
import tqdm
df = pd.read_table(anno_site_file)

records = []
for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
    pdb_id = row['pdb_id']
    uni_id = row['seq_id']
    pdb_posis = [int(i) for i in row['posis'].split(",")]
    pdb_metal_resis = row['metal_resis'].split(',')

    uni_seq_nums = []
    uni_metal_resis = []
    uni_resis = []
    
    if uni_id not in uni_id_to_seq.keys(): continue
    uni_seq = uni_id_to_seq[uni_id]
    pdb_posi_to_uni_posi = align_two_seq(pdb_id_to_seq[pdb_id], uni_seq)

    for idx, posi in enumerate(pdb_posis):
        if posi in pdb_posi_to_uni_posi.keys(): # some may not be aligned
            uni_posi = pdb_posi_to_uni_posi[posi]
            uni_seq_nums.append(uni_posi + 1)
            uni_metal_resis.append(pdb_metal_resis[idx])
            uni_resis.append(uni_seq[uni_posi])

    if len(uni_seq_nums) != 0:
        records.append({
            "seq_id": uni_id,
            "pdb_id": pdb_id,
            "seq_num": uni_seq_nums,
            "resi": uni_resis,
            "metal_resi": uni_metal_resis,
        })

df = pd.DataFrame(records)

100%|██████████| 59949/59949 [02:45<00:00, 362.97it/s]


In [5]:
records = []
for (seq_id,), df_seq in df.groupby(by=["seq_id"]):
    
    resi_records = []
    for _, row in df_seq.iterrows():
        seq_nums = row['seq_num']
        metal_resis = row['metal_resi']
        resis = row['resi']

        for index, s in enumerate(seq_nums):
            resi_records.append({
                "seq_num": s,
                "resi": resis[index],
                "metal_resi": metal_resis[index]
            })
    df_uniq = pd.DataFrame(resi_records).drop_duplicates(subset=["seq_num"]).sort_values(by=["seq_num"])

    records.append({
        "seq_id": seq_id,
        "seq_num": ",".join([str(i) for i in df_uniq["seq_num"]]),
        "resi": ",".join(df_uniq["resi"].tolist()),
        "metal_resi": ",".join(df_uniq["metal_resi"].tolist()),
        "evid_pdb_id": ",".join(df_seq["pdb_id"])
    })

In [6]:
pd.DataFrame(records).to_csv(uni_anno_file, sep="\t", index=None, header=None)